# Making Pandas DataFrames from API Requests
In this example, we will use the U.S. Geological Survey's API to grab a JSON object of earthquake data and convert it to a `pandas.DataFrame`.

USGS API: https://earthquake.usgs.gov/fdsnws/event/1/

### Get Data from API

In [2]:
import datetime as dt
import pandas as pd
import requests

yesterday = dt.date.today() - dt.timedelta(days=1)
api = 'https://earthquake.usgs.gov/fdsnws/event/1/query'
payload = {
    'format': 'geojson',
    'starttime': yesterday - dt.timedelta(days=30),
    'endtime': yesterday
}
response = requests.get(api, params=payload)

# let's make sure the request was OK
response.status_code

200

Response of 200 means OK, so we can pull the data out of the result. Since we asked the API for a JSON payload, we can extract it from the response with the `json()` method.

### Isolate the Data from the JSON Response
We need to check the structures of the response data to know where our data is.

In [3]:
earthquake_json = response.json()
earthquake_json.keys()

dict_keys(['type', 'metadata', 'features', 'bbox'])

The USGS API provides information about our request in the `metadata` key. Note that your result will be different, regardless of the date range you chose, because the API includes a timestamp for when the data was pulled:

In [4]:
earthquake_json['metadata']

{'generated': 1753227126000,
 'url': 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2025-06-21&endtime=2025-07-21',
 'title': 'USGS Earthquakes',
 'status': 200,
 'api': '1.14.1',
 'count': 11138}

Each element in the JSON array `features` is a row of data for our dataframe.

In [5]:
type(earthquake_json['features'])

list

Your data will be different depending on the date you run this.

In [6]:
earthquake_json['features'][0]

{'type': 'Feature',
 'properties': {'mag': 1.2,
  'place': '29 km W of Tyonek, Alaska',
  'time': 1753055919453,
  'updated': 1753056032451,
  'tz': None,
  'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/ak02598t28qx',
  'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=ak02598t28qx&format=geojson',
  'felt': None,
  'cdi': None,
  'mmi': None,
  'alert': None,
  'status': 'automatic',
  'tsunami': 0,
  'sig': 22,
  'net': 'ak',
  'code': '02598t28qx',
  'ids': ',ak02598t28qx,',
  'sources': ',ak,',
  'types': ',origin,phase-data,',
  'nst': None,
  'dmin': None,
  'rms': 0.47,
  'gap': None,
  'magType': 'ml',
  'type': 'earthquake',
  'title': 'M 1.2 - 29 km W of Tyonek, Alaska'},
 'geometry': {'type': 'Point', 'coordinates': [-151.6861, 61.0532, 72.1]},
 'id': 'ak02598t28qx'}

### Convert to DataFrame
We need to grab the `properties` section out of every entry in the `features` JSON array to create our dataframe.

In [7]:
earthquake_properties_data = [
    quake['properties'] for quake in earthquake_json['features']
]
df = pd.DataFrame(earthquake_properties_data)
df.head()

,mag,place,time,updated,tz,url,detail,felt,cdi,mmi,...,ids,sources,types,nst,dmin,rms,gap,magType,type,title
0,1.20,"29 km W of Tyonek, Alaska",1753055919453,1753056032451,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,",ak02598t28qx,",",ak,",",origin,phase-data,",NaN,NaN,0.47,NaN,ml,earthquake,"M 1.2 - 29 km W of Tyonek, Alaska"
1,2.35,"3 km NNE of Tallaboa, Puerto Rico",1753055838260,1753059989440,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,",pr71489888,",",pr,",",origin,phase-data,",7.0,0.12670,0.28,192.0,md,earthquake,"M 2.4 - 3 km NNE of Tallaboa, Puerto Rico"
2,4.50,"168 km E of Petropavlovsk-Kamchatsky, Russia",1753055594092,1753145212040,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,",us7000qe5j,",",us,",",origin,phase-data,",55.0,1.49800,0.76,147.0,mb,earthquake,"M 4.5 - 168 km E of Petropavlovsk-Kamchatsky, ..."
3,0.26,"5 km WSW of Anza, CA",1753055254880,1753110667865,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,NaN,NaN,NaN,...,",ci41027167,",",ci,",",nearby-cities,origin,phase-data,scitech-link,",15.0,0.03517,0.11,66.0,ml,earthquake,"M 0.3 - 5 km WSW of Anza, CA"
4,2.33,"11 km SSW of Cantua Creek, CA",1753054874510,1753057491555,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,1.0,2.0,NaN,...,",nc75212732,",",nc,",",dyfi,focal-mechanism,nearby-cities,origin,pha...",33.0,0.07610,0.26,81.0,md,earthquake,"M 2.3 - 11 km SSW of Cantua Creek, CA"


### (Optional) Write Data to CSV

In [8]:
df.to_csv('earthquakes.csv', index=False)

<hr>
<div>
    <a href="./2-creating_dataframes.ipynb">
        <button style="float: left;">&#8592; Previous Notebook</button>
    </a>
    <a href="./4-inspecting_dataframes.ipynb">
        <button style="float: right;">Next Notebook &#8594;</button>
    </a>
</div>
<br>
<hr>